In [22]:
import numpy as np
import pandas as pd
import gc
from tqdm import tqdm
from os.path import join
import os
import sys
import yaml
from importlib import reload

In [18]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# Tokenizer

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

### Train tokenizer

In [55]:
tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

In [56]:
special_tokens = ["<bos>", "<eos>", "<pad>"]

trainer = BpeTrainer(
    vocab_size=32_000, 
    special_tokens=special_tokens,
    initial_alphabet=ByteLevel.alphabet()
)

In [57]:
dataset = pd.read_csv(
    "/content/drive/MyDrive/machine_translation/datasets/en-es/EN-ES.txt", 
    sep='\t', 
    header = None,
    encoding='utf-8',
)[[0,1]].rename(columns = {0: "EN", 1: "ES"})

In [58]:
corpus = dataset['EN'].dropna().astype(str).tolist() + dataset['ES'].dropna().astype(str).tolist()
del dataset
gc.collect()

131

In [59]:
tokenizer.train_from_iterator(corpus, trainer=trainer)
tokenizer.save("/content/drive/MyDrive/machine_translation/datasets/en-es/bpe.json")

In [61]:
encoded = tokenizer.encode("Hola! Let's check out this custom BPE.")
encoded.ids, encoded.tokens, tokenizer.decode(encoded.ids)

([2697, 269, 3, 8404, 1475, 4815, 852, 633, 4956, 545, 1315, 16],
 ['Ho',
  'la',
  '!',
  'ĠLet',
  "'s",
  'Ġcheck',
  'Ġout',
  'Ġthis',
  'Ġcustom',
  'ĠB',
  'PE',
  '.'],
 "Hola! Let's check out this custom BPE.")

### Encode data using trained tokenizer

In [62]:
dataset = pd.read_csv(
    "/content/drive/MyDrive/machine_translation/datasets/en-es/EN-ES.txt", 
    sep='\t', 
    header = None,
    encoding='utf-8',
)[[0,1]].rename(columns = {0: "EN", 1: "ES"})
dataset = dataset.dropna(how="any")

In [ ]:
tokenizer = Tokenizer.from_file("/content/drive/MyDrive/machine_translation/datasets/en-es/bpe.json")

In [64]:
def get_lengths(dataset, batch_size=10_000):
    lengths = []

    for i in tqdm(range(0, len(dataset), batch_size), desc="Measuring lengths"):
        batch = dataset[i: i + batch_size]

        encoded = tokenizer.encode_batch_fast(batch)

        lengths.extend([len(output.ids) for output in encoded])

        del encoded
        gc.collect()

    return lengths

In [ ]:
lengths_en = np.array(get_lengths(dataset["EN"]), dtype=np.uint16)
lengths_es = np.array(get_lengths(dataset["ES"]), dtype=np.uint16)
lengths = np.concat((lengths_en, lengths_es))

In [68]:
# get length statistics
np.max(lengths), np.percentile(lengths, 99)

(np.uint16(926), np.float64(104.0))

In [74]:
tokenizer = Tokenizer.from_file("/content/drive/MyDrive/machine_translation/datasets/en-es/bpe.json")

max_len = 128
tokenizer.enable_padding(direction="right", pad_id=2, pad_token="<pad>", length=max_len)

In [75]:
# get length mask
length_mask = (lengths_en <= max_len) & (lengths_es <= max_len)
length_mask.sum()

np.int64(5667063)

In [76]:
def encode_batch_and_dump(dataset, filename, batch_size = 10_000):
    n = dataset.shape[0]

    mmap = np.memmap(
        filename,
        dtype=np.uint16,
        mode="w+",
        shape=(n, max_len),
    )

    for start in tqdm(range(0, n, batch_size), desc="Encoding and dumping"):
        end = min(start + batch_size, n)

        batch = dataset[start:end]
        encoded = tokenizer.encode_batch(batch)

        # print([len(output.ids) for output in encoded if len(output.ids) != max_len])

        matrix_slice = np.asarray(
            [output.ids for output in encoded],
            dtype=np.uint16,
        )

        mmap[start:end] = matrix_slice

        del batch, encoded, matrix_slice
        gc.collect()

    mmap.flush()
    del mmap
    gc.collect()

In [77]:
encode_batch_and_dump(dataset.loc[length_mask, "EN"], "/content/drive/MyDrive/machine_translation/datasets/en-es/EN.npy", batch_size = 10_000)

Encoding and dumping: 100%|██████████| 567/567 [14:46<00:00,  1.56s/it]


In [82]:
encode_batch_and_dump(dataset.loc[length_mask, "ES"], "/content/drive/MyDrive/machine_translation/datasets/en-es/ES.npy", batch_size = 10_000)

Encoding and dumping: 100%|██████████| 567/567 [18:30<00:00,  1.96s/it]


# Train model on TPU

In [3]:
!pip install torch-xla

In [21]:
# Make sure all the scripts are in place
# Pull from github

# 1. Define repo details
REPO_URL = "https://github.com/arka816/langlocal.git"
REPO_DIR = "/content/langlocal"

!rm -rf {REPO_DIR}

# 2. Clone if the directory doesn't exist (e.g., after a runtime crash)
if not os.path.exists(REPO_DIR):
    print("Runtime disconnected. Re-cloning scripts...")
    !git clone {REPO_URL}
else:
    # If the runtime didn't crash but you updated code locally, pull the latest changes
    print("Runtime active. Pulling latest script changes...")
    !cd {REPO_DIR} && git pull

# 3. CRITICAL: Add the cloned directory to Python's search path so you can import from it
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

Runtime disconnected. Re-cloning scripts...
Cloning into 'langlocal'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 83 (delta 33), reused 71 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 39.78 KiB | 6.63 MiB/s, done.
Resolving deltas: 100% (33/33), done.


In [31]:
import machine_translation.data_pipeline.collator as collator
reload(collator)

import machine_translation.transformer.train as train
reload(train)

from machine_translation.transformer.train import train_tpu_single_core

In [32]:
config_path = join(REPO_DIR, "machine_translation", "transformer", "config.yaml")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

config

{'data': {'VOCAB_SIZE': 32000, 'SENTENCE_SIZE': 128, 'DATA_SIZE': 5667063},
 'training': {'NUM_EPOCHS': 10, 'BATCH_SIZE': 256},
 'model': {'EMBEDDING_DIM': 512,
  'FFN_DIM': 2048,
  'HEADS': 8,
  'NUM_LAYERS': 6,
  'DROPOUT': 0.1,
  'SHARE_EMBEDDINGS': True}}

In [33]:
src_file = "/content/drive/MyDrive/machine_translation/datasets/en-es/EN.npy"
tgt_file = "/content/drive/MyDrive/machine_translation/datasets/en-es/ES.npy"

data_size = config['data']['DATA_SIZE']
sentence_size = config['data']['SENTENCE_SIZE']
file_dims = (data_size, sentence_size)

model_configs = {
    "src_vocab":    config['data']['VOCAB_SIZE'],
    "tgt_vocab":    config['data']['VOCAB_SIZE'],
    "N":            config['model']['NUM_LAYERS'],
    "d_model":      config['model']['EMBEDDING_DIM'], 
    "d_ff":         config['model'].get("FFN_DIM", 4 * config['model']['EMBEDDING_DIM']), 
    "heads":        config['model']['HEADS'],
    "dropout":      config['model']['DROPOUT'], 
    "share_embedding":  config['model']['SHARE_EMBEDDINGS']
}

epochs = config['training']['NUM_EPOCHS']
batch_size = config['training']['BATCH_SIZE']

checkpoint_filepath =  "/content/drive/MyDrive/machine_translation/en-es-machine-translator.pt"

In [34]:
print("Kicking off training...")
train_tpu_single_core(
    src_file,
    tgt_file,
    file_dims,
    model_configs,
    bos_id=0,
    eos_id=1,
    pad_id=2,
    batch_size=batch_size,
    epochs=epochs,
    loader_workers=0,
    checkpoint_filepath=checkpoint_filepath,
)

Kicking off training...
Using XLA device: xla:0
Number of parameters:  60524544


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:841: UserWarning: aten::std: an autograd kernel was not registered to the Autograd key(s) but we are trying to backprop through it. This may lead to silently incorrect behavior. This behavior is deprecated and will be removed in a future version of PyTorch. If your operator is differentiable, please ensure you have registered an autograd kernel to the correct Autograd key (e.g. DispatchKey::Autograd, DispatchKey::CompositeImplicitAutograd). If your operator is not differentiable, or to squash this warning and use the previous behavior, please register torch::CppFunction::makeFallthrough() to DispatchKey::Autograd. (Triggered internally at /pytorch/torch/csrc/autograd/autograd_not_implemented_fallback.cpp:62.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


ValueError: XLA:TPU compile permanent error. Ran out of memory in memory space hbm. Used 16.00G of 15.75G hbm. Exceeded hbm capacity by 260.38M.

Total hbm usage >= 16.25G:
    reserved        258.00M 
    program          14.94G 
    arguments         1.07G 

Output size 3.51G; shares 1.04G with arguments.

Program hbm requirement 14.94G:
    scoped           289.0K
    HLO temp         14.94G (96.9% utilization: Unpadded (14.33G) Padded (14.79G), 1.0% fragmentation (152.88M))

  Largest program allocations in hbm:

  1. Size: 3.69G
     Shape: f32[256,121,32000]{2,0,1:T(8,128)}
     Unpadded size: 3.69G
     XLA label: fusion.1642.remat3 = fusion(multiply_add_fusion.9, bitcast.4710), kind=kOutput, calls=fused_computation.2952.clone.clone.clone
     Allocation type: HLO temp
     ==========================

  2. Size: 242.00M
     Shape: f32[256,121,2048]{2,0,1:T(8,128)}
     Unpadded size: 242.00M
     XLA label: convolution_add_fusion.2.remat5 = fusion(copy-done.742, custom-call.173